# ⚙️ Step 4: Model Training, Evaluation, and World Cup Simulation
---
This notebook serves as our unified machine learning pipeline. Instead of running a background script, we execute each stage of the data lifecycle here:
1. **Data Ingestion**: Loading and cleaning historical international match data.
2. **Feature Engineering**: Chronological computation of team form, goal differences, and rolling metrics.
3. **Model Selection**: Benchmarking Logistic Regression, Random Forests, and XGBoost using Stratified 5-Fold Cross-Validation.
4. **Simulation**: Using our finalized model to forecast the entire opening round of the FIFA World Cup 2026.

In [1]:
import os
import warnings
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import xgboost as xgb
import joblib

warnings.filterwarnings('ignore')

# Setup project directories dynamically
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
MODEL_DIR = BASE_DIR / "models"
VIS_DIR = BASE_DIR / "visuals"

for folder in [DATA_DIR, MODEL_DIR, VIS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Dataset resource URL
DATA_URL = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"
print("✅ Core engineering libraries loaded and directories verified.")

✅ Core engineering libraries loaded and directories verified.


In [2]:
def load_and_prepare_data() -> pd.DataFrame:
    """Loads historical data from local data folder or downloads it if missing."""
    csv_path = DATA_DIR / "international_results.csv"
    
    if csv_path.exists():
        print("📂 Dataset detected in local cache. Loading...")
        df = pd.read_csv(csv_path)
    else:
        print("📡 Local file missing. Fetching live dataset from remote repository...")
        df = pd.read_csv(DATA_URL)
        df.to_csv(csv_path, index=False)
        print("💾 Download complete. Cached locally.")
            
    # Clean, parse, and filter for modern era matches
    df['date'] = pd.to_datetime(df['date'])
    df = df[df['date'] >= '2000-01-01'].reset_index(drop=True)
    df = df.dropna(subset=['home_score', 'away_score'])
    df['home_score'] = df['home_score'].astype(int)
    df['away_score'] = df['away_score'].astype(int)
    
    print(f"⚽ Successfully prepared {len(df):,} international matches (2000–Present).")
    return df

raw_data = load_and_prepare_data()

📡 Local file missing. Fetching live dataset from remote repository...
💾 Download complete. Cached locally.
⚽ Successfully prepared 25,316 international matches (2000–Present).


In [3]:
class FeatureEngineer:
    """Computes running statistics chronologically to eliminate data leakage."""
    def __init__(self, window: int = 5):
        self.window = window
        self.team_history = {}

    def _update_history(self, team: str, goals_for: int, goals_against: int, pts: int):
        if team not in self.team_history:
            self.team_history[team] = {'gf': [], 'ga': [], 'pts': [], 'wins': []}
        self.team_history[team]['gf'].append(goals_for)
        self.team_history[team]['ga'].append(goals_against)
        self.team_history[team]['pts'].append(pts)
        self.team_history[team]['wins'].append(1 if pts == 3 else 0)

    def _get_form_features(self, team: str, prefix: str) -> dict:
        hist = self.team_history.get(team, {'gf': [], 'ga': [], 'pts': [], 'wins': []})
        recent_gf = hist['gf'][-self.window:]
        recent_ga = hist['ga'][-self.window:]
        recent_pts = hist['pts'][-self.window:]
        recent_wins = hist['wins'][-self.window:]
        
        n = len(recent_gf) if len(recent_gf) > 0 else 1
        return {
            f'{prefix}_avg_goals_last{self.window}': sum(recent_gf) / n,
            f'{prefix}_avg_conceded_last{self.window}': sum(recent_ga) / n,
            f'{prefix}_points_last{self.window}': sum(recent_pts),
            f'{prefix}_win_rate_last{self.window}': sum(recent_wins) / n,
            f'{prefix}_goal_diff_last{self.window}': sum(recent_gf) - sum(recent_ga)
        }

    def build_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.sort_values('date').reset_index(drop=True)
        features_list = []

        for idx, row in df.iterrows():
            home, away = row['home_team'], row['away_team']
            gh, ga = row['home_score'], row['away_score']
            
            # Extract features BEFORE updating state with today's match results
            match_feats = {**self._get_form_features(home, 'home'), **self._get_form_features(away, 'away')}
            features_list.append(match_feats)
            
            h_pts, a_pts = (3, 0) if gh > ga else ((1, 1) if gh == ga else (0, 3))
            self._update_history(home, gh, ga, h_pts)
            self._update_history(away, ga, gh, a_pts)

        return pd.concat([df, pd.DataFrame(features_list)], axis=1)

print("⚙️ Engineering feature transformers...")
engineer = FeatureEngineer(window=5)
engineered_df = engineer.build_features(raw_data)
print("✅ Feature space built successfully.")

⚙️ Engineering feature transformers...
✅ Feature space built successfully.


In [4]:
# Create targets and isolate feature set
conditions = [
    (engineered_df['home_score'] < engineered_df['away_score']),
    (engineered_df['home_score'] == engineered_df['away_score']),
    (engineered_df['home_score'] > engineered_df['away_score'])
]
choices = [0, 1, 2]  # 0: Away Win, 1: Draw, 2: Home Win
engineered_df['outcome'] = np.select(conditions, choices, default=1)

feature_cols = [col for col in engineered_df.columns if 'last5' in col]
X = engineered_df[feature_cols]
y = engineered_df['outcome']

print(f"📊 Dataset shapes: X={X.shape}, y={y.shape}")

📊 Dataset shapes: X=(25316, 10), y=(25316,)


In [5]:
# Benchmark Models using 5-Fold Stratified Cross-Validation
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
    'XGBoost': xgb.XGBClassifier(eval_metric='mlogloss', random_state=42)
}

print("🏋️ Evaluating candidate models...")
for name, model in models.items():
    cv_results = cross_validate(model, X, y, cv=StratifiedKFold(5), scoring=['accuracy', 'f1_macro'])
    print(f" ↳ {name:20} -> Mean Accuracy: {cv_results['test_accuracy'].mean():.4f} | F1-Score: {cv_results['test_f1_macro'].mean():.4f}")

# Finalize and serialize the best-performing model (XGBoost)
print("\n💾 Training final XGBoost engine on complete historical dataset...")
final_model = xgb.XGBClassifier(eval_metric='mlogloss', random_state=42)
final_model.fit(X, y)

model_output_path = MODEL_DIR / "fifa_xgboost.joblib"
joblib.dump(final_model, model_output_path)
print(f"✅ Production model serialized and saved to: {model_output_path}")

🏋️ Evaluating candidate models...
 ↳ Logistic Regression  -> Mean Accuracy: 0.5275 | F1-Score: 0.3660
 ↳ Random Forest        -> Mean Accuracy: 0.5249 | F1-Score: 0.3563
 ↳ XGBoost              -> Mean Accuracy: 0.5125 | F1-Score: 0.3774

💾 Training final XGBoost engine on complete historical dataset...
✅ Production model serialized and saved to: models\fifa_xgboost.joblib


## 📅 World Cup 2026 Simulation
We extract the final operational state of each team at the end of our dataset timeline and use our serialized model to predict the upcoming tournament group match schedule.

In [6]:
# Official 2026 Group Stage Matrix
WC2026_MATCHES = [
    # Group A
    ("Mexico", "South Africa", "2026-06-11"), ("South Korea", "Czechia", "2026-06-11"),
    ("Mexico", "Czechia", "2026-06-17"), ("South Korea", "South Africa", "2026-06-17"),
    ("Mexico", "South Korea", "2026-06-22"), ("Czechia", "South Africa", "2026-06-22"),
    # Group B
    ("Canada", "Bosnia and Herzegovina", "2026-06-12"), ("Qatar", "Switzerland", "2026-06-13"),
    ("Canada", "Qatar", "2026-06-18"), ("Bosnia and Herzegovina", "Switzerland", "2026-06-18"),
    ("Canada", "Switzerland", "2026-06-23"), ("Bosnia and Herzegovina", "Qatar", "2026-06-23"),
    # Group C
    ("Brazil", "Morocco", "2026-06-13"), ("Haiti", "Scotland", "2026-06-13"),
    ("Brazil", "Scotland", "2026-06-18"), ("Morocco", "Haiti", "2026-06-19"),
    ("Brazil", "Haiti", "2026-06-23"), ("Scotland", "Morocco", "2026-06-24"),
    # Group D
    ("USA", "Paraguay", "2026-06-12"), ("Australia", "Turkey", "2026-06-13"),
    ("USA", "Australia", "2026-06-19"), ("Paraguay", "Turkey", "2026-06-19"),
    ("USA", "Turkey", "2026-06-25"), ("Paraguay", "Australia", "2026-06-25")
]

# Fetch the absolute latest historical metrics for each team
latest_state = engineered_df.sort_values('date').groupby('home_team').last().reset_index()

simulation_rows = []
for home, away, date in WC2026_MATCHES:
    # Ensure both teams have record histories in our dataset
    if home in latest_state['home_team'].values and away in latest_state['home_team'].values:
        # Build vector out of latest forms
        h_feats = {f'home_{k.split("home_")[1]}': v for k, v in latest_state[latest_state['home_team'] == home].iloc[0].to_dict().items() if 'home_avg' in k or 'home_point' in k or 'home_win' in k or 'home_goal' in k}
        a_feats = {f'away_{k.split("home_")[1]}': v for k, v in latest_state[latest_state['home_team'] == away].iloc[0].to_dict().items() if 'home_avg' in k or 'home_point' in k or 'home_win' in k or 'home_goal' in k}
        
        match_vector = pd.DataFrame([{**h_feats, **a_feats}])[feature_cols]
        probs = final_model.predict_proba(match_vector)[0]
        
        outcomes = ["Away Win", "Draw", "Home Win"]
        pred_label = outcomes[np.argmax(probs)]
        
        simulation_rows.append({
            "Date": date,
            "Home Team": home,
            "Away Team": away,
            "Away Win Prob": f"{probs[0]*100:.1f}%",
            "Draw Prob": f"{probs[1]*100:.1f}%",
            "Home Win Prob": f"{probs[2]*100:.1f}%",
            "Predicted Outcome": home if pred_label == "Home Win" else (away if pred_label == "Away Win" else "Draw")
        })

sim_df = pd.DataFrame(simulation_rows)
display(sim_df)

,Date,Home Team,Away Team,Away Win Prob,Draw Prob,Home Win Prob,Predicted Outcome
0,2026-06-11,Mexico,South Africa,13.9%,14.5%,71.6%,Mexico
1,2026-06-17,South Korea,South Africa,19.6%,25.4%,55.0%,South Korea
2,2026-06-22,Mexico,South Korea,19.0%,32.0%,49.0%,Mexico
3,2026-06-12,Canada,Bosnia and Herzegovina,24.9%,26.8%,48.4%,Canada
4,2026-06-13,Qatar,Switzerland,40.2%,29.7%,30.1%,Switzerland
5,2026-06-18,Canada,Qatar,5.0%,12.3%,82.7%,Canada
6,2026-06-18,Bosnia and Herzegovina,Switzerland,25.3%,20.0%,54.7%,Bosnia and Herzegovina
7,2026-06-23,Canada,Switzerland,23.6%,20.2%,56.2%,Canada
8,2026-06-23,Bosnia and Herzegovina,Qatar,11.2%,18.0%,70.8%,Bosnia and Herzegovina
9,2026-06-13,Brazil,Morocco,29.8%,39.5%,30.6%,Draw
